## < Training >

### Configuration

In [1]:
#%%
import pandas as pd
import torch
import numpy as np
import argparse
import importlib
from modules.utils import set_random_seed
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torch.utils.data import DataLoader

def get_args(debug):
    parser = argparse.ArgumentParser('parameters')
    
    ### defualt
    parser.add_argument('--seed', type=int, default=0, 
                        help='seed for repeatable results')
    parser.add_argument("--model", type=str, default="SPT")                         
    parser.add_argument('--dataset', type=str, default='anuran', 
                        help="""
                        Tabular dataset options: 
                        banknote, whitewine, breast, bankruptcy, musk,
                        abalone, anuran, shoppers, default, magic
                        """)
    parser.add_argument('--test_size', default=0.2, type=float, 
                        help="Proportion of the dataset to include in the test split")

    ### stage-1 model
    parser.add_argument('--batch_size1', default=512, type=int,                     
                        help="Batch size for stage 1 training")
    parser.add_argument('--var', default=0.1, type=float,                            
                        help='The value of fixed encoder variance') 
    parser.add_argument('--lr1', default=0.001, type=float,
                        help='Learning rate for stage 1 training')
    parser.add_argument('--weight_decay1', default=0, type=float,
                        help='weight decay for stage 1')
    parser.add_argument('--d_token', default=4, type=int,
                        help='Latent dimension')
    parser.add_argument('--num_layers', default=2, type=int,
                        help='The number of layer in transformer')
    parser.add_argument('--epochs', default=4000, type=int,
                        help='Training epochs for stage 1') 
    parser.add_argument('--max_beta', type=float, default=1e-2,                      
                        help='Maximum value of regularization weight')
    parser.add_argument('--min_beta', type=float, default=1e-5, 
                        help='Manimum value of regularization weight')
    parser.add_argument('--lambda', type=float, default=0.7, 
                        help='Initial value of regularization weight')
    parser.add_argument('--factor', default=32, type=int,
                        help='FACTOR')
    parser.add_argument('--n_head', default=1, type=int,
                        help='N_HEAD')
    parser.add_argument('--bias', default=True, type=bool,
                        help='Token Bias')        
    ### stage-2 Model
    parser.add_argument('--batch_size2', default=512, type=int,                     
                        help='Batch size for stage 2 training' )
    parser.add_argument('--scheduler', type=str, default='linear', 
                        help="Options for beta scheduling: linear and cosine")
    parser.add_argument('--weight_decay2', default=1e-4, type=float,
                        help='Weight decay for AdamW in training stage 2')
    parser.add_argument('--epochs_2', default=10_000, type=int,
                        help='Training epochs for stage 2')
    parser.add_argument("--denoising_dim", default=1024, type=int,
                        help="Size of latent dimension for stage 2")
    parser.add_argument("--latent_noise_variance", default=0.0, type=float,                     
                        help="The variance of initial sample")
    
    ### inference factor
    parser.add_argument("--coverage_k", default=5, type=int,
                        help="The nearest neighbor in inference procedure")
    
    if debug:
        return parser.parse_args(args=[])
    else:
        return parser.parse_args()

### Load Dataset and Proprocess

In [2]:
config = vars(get_args(debug=True))
set_random_seed(config['seed'])
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
dataset_module = importlib.import_module(f"datasets.preprocess")
importlib.reload(dataset_module)

CustomDataset = dataset_module.CustomDataset
train_dataset = CustomDataset(
    config, train=True)
train_dataloader = DataLoader(
    train_dataset, batch_size=config['batch_size1'], shuffle=True)

Tranform Categorical Features...: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 648.07it/s]


### Stage 1 (VAE)

In [3]:
model_module = importlib.import_module('modules.model1')
importlib.reload(model_module)

config["d_numerical"] = train_dataset.num_continuous_features
config['categories'] = train_dataset.num_categories # the number of unique value

model1 = model_module.Model_VAE(
    num_layers = config['num_layers'],  
    d_numerical = config["d_numerical"], 
    categories = config['categories'], 
    d_token = config['d_token'],
    var = config['var'],
    factor=config["factor"],
    n_head = config["n_head"],
    bias = config['bias']).to(device) 
model1.train()
optimizer1 = torch.optim.Adam(
    model1.parameters(), 
    lr=config['lr1'],
    weight_decay=config["weight_decay1"])
scheduler1 = ReduceLROnPlateau(optimizer1, mode='min', factor=0.95, patience=10)

# Model Training
train_module1 = importlib.import_module('modules.train1')
importlib.reload(train_module1)
train_z = train_module1.train_function( 
    model1,
    train_dataset,
    train_dataloader,
    config,
    optimizer1, 
    scheduler1,
    device
)

train_z = torch.tensor(train_z).float()
print("<train_z.shape> : ",train_z.shape)

train_z = train_z[:, 1:, :]
B, num_tokens, token_dim = train_z.size()
in_dim = num_tokens * token_dim
train_z = train_z.view(B, in_dim)
print('train_z is formulated by size:',train_z.shape)

[epoch 001], loss: 4.1056, loss_mse: 2.1440, loss_ce: 1.7576, entropy: 20.3972, train_acc: 0.2796
[epoch 101], loss: 0.3192, loss_mse: 0.1442, loss_ce: 0.0223, entropy: 44.5190, train_acc: 0.9919
Learning rate updated: 0.00095
Learning rate updated: 0.0009025
Learning rate updated: 0.000857375
[epoch 201], loss: 0.1171, loss_mse: 0.0572, loss_ce: 0.0021, entropy: 100.2822, train_acc: 1.0000
Learning rate updated: 0.0008145062499999999
Learning rate updated: 0.0007737809374999998
Learning rate updated: 0.0007350918906249997
[epoch 301], loss: 0.0540, loss_mse: 0.0291, loss_ce: 0.0002, entropy: 178.6705, train_acc: 1.0000
Learning rate updated: 0.0006983372960937497
Learning rate updated: 0.0006634204312890621
Learning rate updated: 0.000630249409724609
[epoch 401], loss: 0.0266, loss_mse: 0.0155, loss_ce: 0.0001, entropy: 333.5815, train_acc: 1.0000
Learning rate updated: 0.0005987369392383785
Learning rate updated: 0.0005688000922764595
Learning rate updated: 0.0005403600876626365
Lear

### Stage 2 (Diffusion)

In [4]:
# Stage 2 (Diffusion) 
denoise_fn_module = importlib.import_module(f"modules.model2")
importlib.reload(denoise_fn_module)
denoise_fn = denoise_fn_module.MLPDiffusion(
    in_dim,
    config['denoising_dim'], ## fixed 1024
).to(device)
denoise_fn.train()

model2 = denoise_fn_module.Model(
    denoise_fn = denoise_fn, hid_dim = in_dim
).to(device)

count_parameters = lambda model: sum(p.numel() for p in model.parameters() if p.requires_grad)
num_params_1 = count_parameters(model1) 
num_params_2 = count_parameters(model2)
print(f"Number of VAE Parameters: {num_params_1 / 1_000_000:.6f}M")
print(f"Number of Diffusion Parameters: {num_params_2 / 1_000_000:.6f}M")

optimizer2 = torch.optim.Adam(model2.parameters(), lr=1e-3, weight_decay=0)  
scheduler2 = ReduceLROnPlateau(optimizer2, mode='min', factor=0.9, patience=20)
model2.train()

# Model Training
train_module2 = importlib.import_module('modules.train2')
importlib.reload(train_module2)
print("TRAIN model2")
train_module2.train_function(
    config,
    model2,
    train_z,
    optimizer2,
    scheduler2,
    device,
)


Number of VAE Parameters: 0.005446M
Number of Diffusion Parameters: 10.698852M
TRAIN model2
[Epoch    0/10000] diffusion_loss: 1.477345
[Epoch    1/10000] diffusion_loss: 1.217423
[Epoch  500/10000] diffusion_loss: 0.381905
[Epoch 1000/10000] diffusion_loss: 0.313444
[Epoch 1500/10000] diffusion_loss: 0.301711
[Epoch 2000/10000] diffusion_loss: 0.304241
[Epoch 2500/10000] diffusion_loss: 0.303604
Early stopping


## < Inference >

In [5]:
from datasets.preprocess import build_num_inverse_fn, build_cat_inverse_fn
from modules.utils import noise_sample, recover_data, split_num_cat_target
from evaluation.clipped_coverage import ClippedDensityCoverage 
from prdc import compute_prdc
from dython.nominal import associations
from synthetic_eval import evaluation

CustomDataset = dataset_module.CustomDataset
train_dataset = CustomDataset(
    config, train=True)
test_dataset = CustomDataset(
    config, train='test', cont_scalers=train_dataset.cont_scalers, cat_scalers=train_dataset.cat_scalers )        
num_inverse = build_num_inverse_fn(train_dataset.cont_scalers)
cat_inverse = build_cat_inverse_fn(train_dataset.cat_scalers)

model2.eval()

model_name = f"{config['dataset']}_{config['lr1']}_{config['d_token']}_{config['denoising_dim']}"
model_name += f"_{config['batch_size1']}_{config['batch_size2']}_{config['max_beta']}"
model_name += f"_{config['var']}_{config['lambda']}"

info = train_dataset.info
info['model_dir'] = f"./assets/models/{model_name}/stage1_{model_name}_{config['seed']}.pth"

config["d_numerical"] = train_dataset.num_continuous_features
config['categories'] = train_dataset.num_categories # the number of unique value


Tranform Categorical Features...: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 584.44it/s]


### Synthetic Data Generation

In [6]:
num_samples = B
    
x_next = noise_sample(model2.denoise_fn_D, num_samples, in_dim, config['latent_noise_variance'])
x_next = x_next * 2 + train_z.mean(0).to(device)

syn_data = x_next.float().cpu().numpy()
syn_data = syn_data.astype(np.float32)
#%%
syn_num, syn_cat, syn_target = split_num_cat_target(syn_data, info, num_inverse, cat_inverse, config, device) 

syn_df = recover_data(syn_num, syn_cat, syn_target, info)

idx_name_mapping = info['idx_name_mapping']
idx_name_mapping = {int(key): value for key, value in idx_name_mapping.items()}

syn_df.rename(columns = idx_name_mapping, inplace=True)
syn_df[train_dataset.categorical_features] = syn_df[train_dataset.categorical_features].astype(int)
syn_df[train_dataset.integer_features] = syn_df[train_dataset.integer_features].astype(int)
syn_df[train_dataset.continuous_features] = syn_df[train_dataset.continuous_features].astype(np.float32)

number of numerical features 22


### Evaluation

In [7]:
""" Synthetic Eval packages """
results = evaluation.evaluate(
    syn_df,train_dataset.raw_data.astype('float32'), test_dataset.raw_data.astype('float32'), 
    train_dataset.ClfTarget, train_dataset.continuous_features, train_dataset.categorical_features, device
    )
'''print results'''
for x, y in results._asdict().items():
    print(f"{x}: {y:.3f}")

results_ = results._asdict().copy()

# Coverage
print("Computing Coverage...")
coverage_k = config['coverage_k']
coverage_res = compute_prdc(real_features=train_dataset.raw_data.astype('float32'),
                            fake_features=syn_df.astype('float32'),
                            nearest_k=coverage_k)
results_['Coverage'] = coverage_res["coverage"]

# Clipped Coverage
print("Computing Clipped Coverage...")
CDC = ClippedDensityCoverage(train_dataset.raw_data.to_numpy(dtype='float32'),
                            K=coverage_k,
                            n_jobs=8,)
results_['Clipped_Coverage'] = CDC.ClippedCoverage(syn_df.to_numpy(dtype='float32'))

# PCD (Pairwise Correlation Difference)
print("Computing PCD...")
syn_asso = associations(syn_df, nominal_columns=train_dataset.categorical_features, compute_only=True)
true_asso = associations(train_dataset.raw_data, nominal_columns=train_dataset.categorical_features, compute_only=True)
pcd_corr = np.linalg.norm(true_asso["corr"] - syn_asso["corr"])
results_['PCD'] = pcd_corr




1. Statistical Fidelity

(marginal) KL-Divergence...

(marginal) Goodness Of Fit...

(joint) MMD...

(joint) Cramer-Wold Distance...

(joint) alpha-precision, beta-recall...


2. Machine Learning Utility

Classification downstream task...

(Baseline) Classification: Accuracy...
[logit] ACC: 0.996
[KNN] ACC: 0.990
[RBF-SVM] ACC: 0.993
[RandomForest] ACC: 0.998
[GradBoost] ACC: 0.999
[AdaBoost] ACC: 0.875
(Synthetic) Classification: Accuracy...
[logit] ACC: 0.993
[KNN] ACC: 0.990
[RBF-SVM] ACC: 0.993
[RandomForest] ACC: 0.997
[GradBoost] ACC: 0.995
[AdaBoost] ACC: 0.725

3. Privacy Preservability

K-anonimity...

K-Map...

Distance to Closest Record...

Attribute Disclosure...

KL: 0.015
GoF: 0.017
MMD: 0.003
CW: 0.009
alpha_precision: 0.970
beta_recall: 0.725
base_cls: 0.975
syn_cls: 0.949
model_selection: 0.928
feature_selection: 0.734
Kanon_base: 0.851
Kanon_syn: 1.286
KMap: 0.886
DCR_RS: 0.060
DCR_RR: 0.063
DCR_SS: 0.062
AD: 0.917
Computing Coverage...
Num real: 5756 Num fake: 5756


In [8]:
metrics = ['GoF','MMD','PCD','alpha_precision','beta_recall','Coverage','Clipped_Coverage',"syn_cls", "model_selection", "feature_selection"]

for metric in metrics:
    val = results_[metric]
    print(f"{metric} : {val:.3f}")

GoF : 0.017
MMD : 0.003
PCD : 0.782
alpha_precision : 0.970
beta_recall : 0.725
Coverage : 0.987
Clipped_Coverage : 1.000
syn_cls : 0.949
model_selection : 0.928
feature_selection : 0.734
